# Modeling Baseline and Nuisance Effects

This notebook demonstrates how to model baseline drift and nuisance effects in fMRI data using pyfmridesign.

Author: Bradley R. Buchsbaum (Python port)  
Date: 2024

## The Purpose of a Baseline Model

In fMRI analysis, the BOLD signal contains not only task-related activity but also various sources of noise and drift. A **baseline model** aims to capture and account for this non-neuronal variance, ensuring that estimates of task effects are more accurate.

These baseline regressors typically model low-frequency scanner drift (using basis functions) and other known sources of noise like head motion parameters or physiological fluctuations.

`pyfmridesign` uses the `baseline_model()` function to construct this part of the design matrix.

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats

from pyfmridesign import (
    baseline_model, baseline, nuisance, block,
    SamplingFrame, design_matrix, Poly, BSpline
)

# Set up plotting style
plt.style.use('seaborn-v0_8-darkgrid')

## Modeling Drift with Basis Functions

A common approach to modeling slow scanner drift is to include a set of basis functions in the model for each run. `pyfmridesign` supports several basis sets:

- `"poly"`: Polynomial functions (orthogonalized)
- `"bs"`: B-spline basis functions
- `"ns"`: Natural spline basis functions
- `"constant"`: Includes only an intercept term for each run

These basis functions are generated separately for each scanning run defined in the `sampling_frame`.

In [ ]:
# Define a sampling frame for two runs of 100 scans each, TR=2s
TR = 2.0
sframe_raw = SamplingFrame(blocklens=[100, 100], tr=TR)

# Create polynomial baseline model (default)
baseline_poly = baseline_model(sframe_raw, basis="poly", degree=3)

# Get the design matrix
X_poly = design_matrix(baseline_poly)

print(f"Baseline design matrix shape: {X_poly.shape}")
print(f"Number of basis functions per run: {X_poly.shape[1] // 2}")

# Visualize the polynomial basis functions
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Run 1
time_run1 = np.arange(100) * TR
for i in range(4):  # degree 3 + intercept = 4 basis functions
    ax1.plot(time_run1, X_poly[:100, i], linewidth=2, label=f'Poly {i}')
ax1.set_xlabel('Time (seconds)')
ax1.set_ylabel('Basis Function Value')
ax1.set_title('Run 1: Polynomial Basis (degree=3)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Run 2
time_run2 = np.arange(100, 200) * TR
for i in range(4):
    ax2.plot(time_run2, X_poly[100:, i+4], linewidth=2, label=f'Poly {i}')
ax2.set_xlabel('Time (seconds)')
ax2.set_ylabel('Basis Function Value')
ax2.set_title('Run 2: Polynomial Basis (degree=3)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## B-spline Basis Functions

B-splines offer more localized basis functions compared to polynomials, which can be advantageous for modeling complex drift patterns:

In [ ]:
# Create B-spline baseline model
baseline_bs = baseline_model(sframe_raw, basis="bs", degree=5)

# Get the design matrix
X_bs = design_matrix(baseline_bs)

print(f"B-spline design matrix shape: {X_bs.shape}")
print(f"Number of B-spline basis functions per run: {X_bs.shape[1] // 2}")

# Visualize B-spline basis functions for run 1
plt.figure(figsize=(12, 6))
time_points = np.arange(100) * TR

for i in range(X_bs.shape[1] // 2):
    plt.plot(time_points, X_bs[:100, i], linewidth=2, label=f'B-spline {i+1}')

plt.xlabel('Time (seconds)')
plt.ylabel('Basis Function Value')
plt.title('B-spline Basis Functions (degree=5)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Including Nuisance Variables

Real fMRI analyses often need to include additional nuisance regressors such as motion parameters, physiological signals, or other confounds. The `baseline_model()` function can incorporate these:

In [ ]:
# Simulate motion parameters (6 parameters x 200 time points)
np.random.seed(42)
motion_params = pd.DataFrame({
    'x_trans': np.cumsum(np.random.randn(200) * 0.1),
    'y_trans': np.cumsum(np.random.randn(200) * 0.1),
    'z_trans': np.cumsum(np.random.randn(200) * 0.1),
    'x_rot': np.cumsum(np.random.randn(200) * 0.01),
    'y_rot': np.cumsum(np.random.randn(200) * 0.01),
    'z_rot': np.cumsum(np.random.randn(200) * 0.01)
})

# Add a block indicator
motion_params['block'] = np.repeat([1, 2], 100)

# Create nuisance specifications
nuisance_vars = [
    nuisance(motion_params[['x_trans', 'y_trans', 'z_trans']]),
    nuisance(motion_params[['x_rot', 'y_rot', 'z_rot']])
]

# Create baseline model with drift and nuisance regressors
baseline_with_nuisance = baseline_model(
    sframe_raw,
    basis="poly",
    degree=3,
    nuisance=nuisance_vars
)

# Get the design matrix
X_nuisance = design_matrix(baseline_with_nuisance)

print(f"Design matrix with nuisance variables: {X_nuisance.shape}")
print(f"Breakdown:")
print(f"  - Polynomial basis: {4 * 2} columns (4 per run)")
print(f"  - Motion parameters: 6 columns")
print(f"  - Total: {X_nuisance.shape[1]} columns")

# Visualize the design matrix
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Polynomial basis for both runs
ax = axes[0, 0]
im1 = ax.imshow(X_nuisance[:, :8], aspect='auto', cmap='RdBu_r', interpolation='nearest')
ax.set_title('Polynomial Basis Functions')
ax.set_xlabel('Basis Function')
ax.set_ylabel('Time Point')
plt.colorbar(im1, ax=ax)

# Motion parameters
ax = axes[0, 1]
im2 = ax.imshow(X_nuisance[:, 8:], aspect='auto', cmap='RdBu_r', interpolation='nearest')
ax.set_title('Motion Parameters')
ax.set_xlabel('Parameter')
ax.set_ylabel('Time Point')
ax.set_xticks(range(6))
ax.set_xticklabels(['X', 'Y', 'Z', 'Pitch', 'Roll', 'Yaw'])
plt.colorbar(im2, ax=ax)

# Plot motion timeseries
ax = axes[1, 0]
time_points = np.arange(200) * TR
ax.plot(time_points, motion_params['x_trans'], label='X translation')
ax.plot(time_points, motion_params['y_trans'], label='Y translation')
ax.plot(time_points, motion_params['z_trans'], label='Z translation')
ax.set_xlabel('Time (seconds)')
ax.set_ylabel('Translation (mm)')
ax.set_title('Translation Parameters')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot rotation timeseries
ax = axes[1, 1]
ax.plot(time_points, motion_params['x_rot'], label='Pitch')
ax.plot(time_points, motion_params['y_rot'], label='Roll')
ax.plot(time_points, motion_params['z_rot'], label='Yaw')
ax.set_xlabel('Time (seconds)')
ax.set_ylabel('Rotation (radians)')
ax.set_title('Rotation Parameters')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## High-pass Filtering with Basis Functions

The choice of basis functions and their parameters effectively implements high-pass filtering. Let's examine the frequency response:

In [ ]:
# Create different baseline models with varying degrees
degrees = [1, 3, 5, 10]
baseline_models = {}

for deg in degrees:
    baseline_models[f'Poly {deg}'] = baseline_model(
        SamplingFrame(n_scans=200, tr=TR),
        basis="poly",
        degree=deg
    )

# Simulate the filtering effect
# Create a signal with multiple frequency components
time = np.arange(200) * TR
signal = (
    np.sin(2 * np.pi * 0.01 * time) +  # Very slow drift
    0.5 * np.sin(2 * np.pi * 0.05 * time) +  # Slow oscillation
    0.3 * np.sin(2 * np.pi * 0.1 * time) +  # Task frequency
    0.2 * np.sin(2 * np.pi * 0.2 * time)  # Higher frequency
)

# Plot original and filtered signals
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for idx, (name, model) in enumerate(baseline_models.items()):
    # Get design matrix
    X = design_matrix(model)
    
    # Project out the baseline (high-pass filter)
    # Residual = signal - X @ (X'X)^-1 @ X' @ signal
    beta = np.linalg.lstsq(X, signal, rcond=None)[0]
    fitted_baseline = X @ beta
    filtered_signal = signal - fitted_baseline
    
    ax = axes[idx]
    ax.plot(time, signal, 'gray', alpha=0.5, linewidth=2, label='Original')
    ax.plot(time, fitted_baseline, 'red', linewidth=2, label='Fitted baseline')
    ax.plot(time, filtered_signal, 'blue', linewidth=2, label='Filtered signal')
    
    ax.set_xlabel('Time (seconds)')
    ax.set_ylabel('Signal Amplitude')
    ax.set_title(name)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Effect of Polynomial Degree on High-pass Filtering', fontsize=16)
plt.tight_layout()
plt.show()

## Block-specific Intercepts

For multi-run experiments, it's important to model run-specific means. This is automatically handled by the basis functions, but we can also explicitly model block effects:

In [ ]:
# Create a 3-run experiment with different baseline levels
sframe_3runs = SamplingFrame(blocklens=[80, 80, 80], tr=TR)

# Simulate data with different baseline levels per run
np.random.seed(42)
run_means = [100, 110, 95]  # Different baseline signal per run
data_3runs = np.concatenate([
    np.random.randn(80) * 5 + mean for mean in run_means
])

# Create baseline model
baseline_3runs = baseline_model(
    sframe_3runs,
    basis="poly",
    degree=2
)

# Get design matrix and fit
X_3runs = design_matrix(baseline_3runs)
beta_3runs = np.linalg.lstsq(X_3runs, data_3runs, rcond=None)[0]
fitted_3runs = X_3runs @ beta_3runs

# Visualize
plt.figure(figsize=(14, 6))
time_points = np.arange(240) * TR

# Plot data and fits
plt.plot(time_points, data_3runs, 'gray', alpha=0.5, linewidth=1, label='Data')
plt.plot(time_points, fitted_3runs, 'red', linewidth=2, label='Fitted baseline')

# Mark run boundaries
run_boundaries = np.cumsum([0, 80, 80, 80]) * TR
for boundary in run_boundaries[1:-1]:
    plt.axvline(x=boundary, color='black', linestyle='--', alpha=0.5)

# Add run labels
for i, (start, end) in enumerate(zip(run_boundaries[:-1], run_boundaries[1:])):
    plt.text((start + end) / 2, 115, f'Run {i+1}\nMean={run_means[i]}', 
             ha='center', va='center', fontsize=10, 
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.xlabel('Time (seconds)')
plt.ylabel('Signal')
plt.title('Baseline Model with Run-specific Intercepts')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Show intercept estimates
print("\nEstimated parameters (first coefficient per run is intercept):")
for i in range(3):
    run_params = beta_3runs[i*3:(i+1)*3]
    print(f"Run {i+1}: Intercept={run_params[0]:.2f}, "
          f"Linear={run_params[1]:.2e}, Quadratic={run_params[2]:.2e}")

## Combining Baseline Models with Task Regressors

In practice, the baseline model is combined with task regressors. Let's see a complete example:

In [ ]:
from pyfmridesign import (
    event_model, EventFactor, HRF_SPMG1,
    design_map
)

# Create experimental data
n_trials = 40
trial_data = pd.DataFrame({
    'onset': np.sort(np.random.uniform(0, 380, n_trials)),
    'condition': np.random.choice(['A', 'B'], n_trials),
    'block': np.repeat([1, 2], n_trials // 2)
})

# Create sampling frame
sframe = SamplingFrame(blocklens=[100, 100], tr=2.0)

# Create event model
emodel = event_model(
    onset ~ condition,
    data=trial_data,
    block="block",
    sampling_frame=sframe,
    hrf=HRF_SPMG1
)

# Create baseline model with nuisance
bmodel = baseline_model(
    sframe,
    basis="poly",
    degree=3
)

# Get full design matrices
X_task = design_matrix(emodel)
X_baseline = design_matrix(bmodel)

# Combine into full design matrix
X_full = np.column_stack([X_task, X_baseline])

print(f"Task design matrix: {X_task.shape}")
print(f"Baseline design matrix: {X_baseline.shape}")
print(f"Full design matrix: {X_full.shape}")

# Visualize the full design matrix
plt.figure(figsize=(12, 8))

# Create custom colormap boundaries
vmax = np.max(np.abs(X_full))
im = plt.imshow(X_full, aspect='auto', cmap='RdBu_r', 
                interpolation='nearest', vmin=-vmax, vmax=vmax)

# Add labels and separators
plt.axvline(x=X_task.shape[1] - 0.5, color='black', linewidth=2)
plt.text(X_task.shape[1] // 2, -5, 'Task Regressors', 
         ha='center', fontsize=12, fontweight='bold')
plt.text(X_task.shape[1] + X_baseline.shape[1] // 2, -5, 'Baseline Regressors', 
         ha='center', fontsize=12, fontweight='bold')

plt.xlabel('Regressors')
plt.ylabel('Time Points (scans)')
plt.title('Complete Design Matrix: Task + Baseline')
plt.colorbar(im, label='Regressor Value')
plt.tight_layout()
plt.show()

## Summary

In this notebook, we've covered:

1. **Baseline models** - Modeling scanner drift and other baseline effects
2. **Basis functions** - Different types (polynomial, B-spline) and their properties
3. **Nuisance variables** - Including motion parameters and other confounds
4. **High-pass filtering** - How basis functions effectively filter the data
5. **Multi-run experiments** - Handling run-specific baselines
6. **Complete models** - Combining task and baseline regressors

Key takeaways:
- Proper baseline modeling is crucial for accurate task effect estimation
- The choice of basis functions affects the filtering properties
- Higher polynomial degrees or more splines allow modeling of faster drifts
- Nuisance regressors should be included when available
- Each run should have its own set of baseline regressors